# AnalystLab Africa ML Internship — Week 6
## Model Tuning & Validation
**Dataset:** Pima Indians Diabetes Database  
**Author:** ML Intern  
**Tools:** Python · Pandas · Scikit-learn · Matplotlib · Seaborn

## Step 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, StratifiedKFold,
                                     learning_curve, validation_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("✅ Libraries loaded.")

## Step 2 — Data Loading & Preprocessing

### Dataset Description
The Pima Indians Diabetes dataset contains diagnostic measurements for 768 female
patients of Pima Indian heritage. The goal is to predict whether a patient has
diabetes (`Outcome` = 1) or not (`Outcome` = 0) based on 8 medical predictor variables.

In [ ]:
df = pd.read_csv('diabetes.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print(f"\nOutcome distribution:")
print(df['Outcome'].value_counts())
print(f"Diabetes prevalence: {df['Outcome'].mean()*100:.1f}%")

In [ ]:
# IMPORTANT: Several features use 0 to represent missing data, which is
# biologically impossible (e.g. BloodPressure=0, BMI=0)
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
print("Zero values (likely missing data) per column:")
for c in zero_cols:
    n_zero = (df[c]==0).sum()
    print(f"  {c:20} {n_zero:4} ({n_zero/len(df)*100:.1f}%)")

In [ ]:
# Replace zeros with NaN, then impute with median
for c in zero_cols:
    df[c] = df[c].replace(0, np.nan)

print("Missing values after conversion:")
print(df[zero_cols].isnull().sum())

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(df.drop(columns=['Outcome'])), columns=df.columns[:-1])
y = df['Outcome']
print(f"\n✅ Missing values imputed with median. Final missing count: {X.isnull().sum().sum()}")

In [ ]:
# EDA visualisations
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df['Outcome'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#22c55e','#ef4444'], rot=0)
axes[0,0].set_title('Outcome Distribution (0=No Diabetes, 1=Diabetes)')
axes[0,0].set_ylabel('Count')

missing_pct = pd.Series({c: (df[c]==np.nan).sum() for c in zero_cols})
missing_pct = (X.isnull().sum() / len(df) * 100)
missing_orig = pd.Series({c: 0 for c in X.columns})
for c in zero_cols:
    raw = pd.read_csv('diabetes.csv')
    missing_orig[c] = (raw[c]==0).sum()/len(raw)*100
missing_orig[missing_orig>0].sort_values(ascending=False).plot(kind='barh', ax=axes[0,1], color='#f97316')
axes[0,1].set_title('Missing Data (encoded as 0) by Feature')
axes[0,1].set_xlabel('Missing %')

axes[1,0].hist(df['Glucose'].dropna(), bins=30, color='#2563eb', edgecolor='white')
axes[1,0].set_title('Glucose Distribution')
axes[1,0].set_xlabel('Glucose')

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', ax=axes[1,1], linewidths=0.3)
axes[1,1].set_title('Feature Correlation Heatmap')
plt.suptitle('Pima Indians Diabetes — EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('week6_eda.png', bbox_inches='tight')
plt.show()

## Step 3 — Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples: {len(X_train)} (80%)")
print(f"Testing samples:  {len(X_test)} (20%)")
print(f"Stratification check — Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Step 4 — Baseline Model

A Random Forest with default parameters serves as our baseline. All subsequent
tuning will be measured against this baseline.

In [ ]:
def evaluate(name, model, Xtr, Xte, ytr, yte, verbose=True):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    yp_prob = model.predict_proba(Xte)[:,1]
    res = {
        'Accuracy':  round(accuracy_score(yte,yp)*100,2),
        'Precision': round(precision_score(yte,yp)*100,2),
        'Recall':    round(recall_score(yte,yp)*100,2),
        'F1':        round(f1_score(yte,yp)*100,2),
        'ROC_AUC':   round(roc_auc_score(yte,yp_prob)*100,2)
    }
    if verbose:
        print(f"{name}:")
        for k, v in res.items():
            print(f"  {k:10}: {v}%")
    return res, model

baseline = RandomForestClassifier(random_state=42, n_jobs=-1)
baseline_res, baseline_model = evaluate('Baseline Random Forest (Test Set)',
                                         baseline, X_train_sc, X_test_sc, y_train, y_test)

In [ ]:
# Baseline cross-validation
cv_scores_baseline = cross_val_score(baseline, X_train_sc, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)
print(f"\nBaseline 5-Fold CV Accuracy: {cv_scores_baseline.mean()*100:.2f}% +/- {cv_scores_baseline.std()*100:.2f}%")
print(f"Fold scores: {[round(s*100,2) for s in cv_scores_baseline]}")

## Step 5 — Hyperparameter Tuning

### 5.1 Grid Search
GridSearchCV exhaustively tries every combination of specified parameters.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}
print(f"Grid size: {2*4*2*2} combinations x 5 folds = "
      f"{2*4*2*2*5} model fits")

grid_search = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1),
                           param_grid, cv=cv5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_sc, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_*100:.2f}%")

In [ ]:
grid_res, grid_model = evaluate('Grid Search Tuned (Test Set)',
                                 grid_search.best_estimator_, X_train_sc, X_test_sc, y_train, y_test)

### 5.2 Random Search
RandomizedSearchCV samples a fixed number of random combinations from the
parameter distributions — more efficient for large search spaces.

In [ ]:
param_dist = {
    'n_estimators': np.arange(50, 350, 25),
    'max_depth': [None] + list(np.arange(3, 16)),
    'min_samples_split': np.arange(2, 12),
    'min_samples_leaf': np.arange(1, 6),
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=1),
                                    param_dist, n_iter=30, cv=cv5,
                                    scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(X_train_sc, y_train)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_*100:.2f}%")

In [ ]:
random_res, random_model = evaluate('Random Search Tuned (Test Set)',
                                     random_search.best_estimator_, X_train_sc, X_test_sc, y_train, y_test)

In [ ]:
# Before vs After comparison (test set)
metric_names = ['Accuracy','Precision','Recall','F1 Score','ROC-AUC']
before = list(baseline_res.values())
after  = list(grid_res.values())

x = np.arange(len(metric_names))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width/2, before, width, label='Baseline', color='#94a3b8')
ax.bar(x + width/2, after, width, label='Grid-Tuned', color='#2563eb')
ax.set_xticks(x); ax.set_xticklabels(metric_names)
ax.set_ylabel('Score (%)')
ax.set_title('Test Set Performance: Before vs After Tuning', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend()
for i, (b, a) in enumerate(zip(before, after)):
    ax.text(i-width/2, b+1, f'{b}%', ha='center', fontsize=9)
    ax.text(i+width/2, a+1, f'{a}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('week6_before_after.png', bbox_inches='tight')
plt.show()
print("Note: test-set metrics can be noisy on a small dataset (154 test samples).")
print("Cross-validation scores below give a more reliable picture.")

## Step 6 — Cross-Validation Implementation

### 6.1 Comparing Models with 5-Fold CV

In [ ]:
cv_results = {}
for name, model in [('Baseline', baseline_model), ('Grid Search', grid_model), ('Random Search', random_model)]:
    cv = cross_val_score(model, X_train_sc, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)
    cv_results[name] = (cv.mean()*100, cv.std()*100)
    print(f"{name:15}: {cv.mean()*100:.2f}% +/- {cv.std()*100:.2f}%")

fig, ax = plt.subplots(figsize=(9, 5))
names = list(cv_results.keys())
means = [v[0] for v in cv_results.values()]
stds  = [v[1] for v in cv_results.values()]
colors = ['#94a3b8','#2563eb','#22c55e']
bars = ax.bar(names, means, yerr=stds, color=colors, capsize=8, alpha=0.85)
ax.set_ylim(65, 90)
ax.set_ylabel('5-Fold CV Accuracy (%)')
ax.set_title('Cross-Validation Accuracy: Baseline vs Tuned Models', fontweight='bold')
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, m+2.5, f'{m:.2f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('week6_cv_comparison.png', bbox_inches='tight')
plt.show()

### 6.2 Effect of K in K-Fold Cross-Validation

In [ ]:
k_results = {}
for k in [3, 5, 10]:
    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    cv = cross_val_score(grid_model, X_train_sc, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
    k_results[k] = (cv.mean()*100, cv.std()*100)
    print(f"K={k:2}: {cv.mean()*100:.2f}% +/- {cv.std()*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
ks = list(k_results.keys())
means_k = [v[0] for v in k_results.values()]
stds_k  = [v[1] for v in k_results.values()]
ax.errorbar(ks, means_k, yerr=stds_k, marker='o', markersize=10, color='#2563eb',
            capsize=8, linewidth=2, capthick=2)
ax.set_xticks(ks)
ax.set_xlabel('Number of Folds (K)')
ax.set_ylabel('CV Accuracy (%)')
ax.set_title('Effect of K in K-Fold Cross-Validation', fontweight='bold')
for k, m, s in zip(ks, means_k, stds_k):
    ax.annotate(f'{m:.2f}% ± {s:.2f}%', (k, m), textcoords="offset points", xytext=(15,5), fontsize=10)
plt.tight_layout()
plt.savefig('week6_kfold.png', bbox_inches='tight')
plt.show()
print("Lower K = larger validation folds, fewer training samples, lower variance estimate.")
print("Higher K = more training data per fold, but higher variance across folds.")

### 6.3 ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model, color in [('Baseline', baseline_model, '#94a3b8'), ('Tuned (Grid Search)', grid_model, '#2563eb')]:
    yp_prob = model.predict_proba(X_test_sc)[:,1]
    fpr, tpr, _ = roc_curve(y_test, yp_prob)
    auc = roc_auc_score(y_test, yp_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Baseline vs Tuned Model', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('week6_roc.png', bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (name, model) in zip(axes, [('Baseline', baseline_model), ('Tuned (Grid Search)', grid_model)]):
    yp = model.predict(X_test_sc)
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=['No Diabetes','Diabetes']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAccuracy: {accuracy_score(y_test,yp)*100:.2f}%')
plt.suptitle('Confusion Matrices — Before vs After Tuning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('week6_confusion.png', bbox_inches='tight')
plt.show()

## Step 7 — Bias-Variance Tradeoff

### Concept Explanation

| | Underfitting (High Bias) | Good Fit | Overfitting (High Variance) |
|---|---|---|---|
| Description | Model too simple | Balanced complexity | Model memorises training data |
| Train Accuracy | Low | High | Very High |
| Validation Accuracy | Low | High | Low |
| Cause | Too few features, too simple a model | Appropriate complexity | Too complex, too many parameters |

**Strategies for balancing bias and variance:**
- Increase model complexity to reduce bias (deeper trees, more features)
- Use regularisation, pruning, or ensemble averaging to reduce variance
- Use cross-validation to detect overfitting before it affects test performance
- Gather more training data to reduce variance without increasing bias

In [ ]:
# Validation curve: max_depth vs accuracy (demonstrates bias-variance tradeoff)
depth_range = np.arange(1, 21)
train_scores, val_scores = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_sc, y_train, param_name='max_depth', param_range=depth_range,
    cv=cv5, scoring='accuracy', n_jobs=-1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depth_range, train_scores.mean(axis=1)*100, 'o-', color='#2563eb', label='Training Accuracy')
ax.fill_between(depth_range, (train_scores.mean(axis=1)-train_scores.std(axis=1))*100,
                (train_scores.mean(axis=1)+train_scores.std(axis=1))*100, alpha=0.1, color='#2563eb')
ax.plot(depth_range, val_scores.mean(axis=1)*100, 's-', color='#ef4444', label='Validation Accuracy (CV)')
ax.fill_between(depth_range, (val_scores.mean(axis=1)-val_scores.std(axis=1))*100,
                (val_scores.mean(axis=1)+val_scores.std(axis=1))*100, alpha=0.1, color='#ef4444')
best_depth = depth_range[np.argmax(val_scores.mean(axis=1))]
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, label=f'Best depth={best_depth}')
ax.set_xlabel('max_depth (Model Complexity)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Bias-Variance Tradeoff: max_depth vs Accuracy', fontweight='bold')
ax.legend()
ax.annotate('Underfitting\n(High Bias)', xy=(2, 70), fontsize=10, color='purple')
ax.annotate('Overfitting\n(High Variance)', xy=(14, 85), fontsize=10, color='red')
plt.tight_layout()
plt.savefig('week6_bias_variance.png', bbox_inches='tight')
plt.show()
print(f"Optimal max_depth: {best_depth} — beyond this, the gap between")
print("training and validation accuracy widens, indicating overfitting.")

## Step 8 — Final Optimized Model

In [ ]:
final_model = grid_search.best_estimator_
final_res, _ = evaluate('FINAL OPTIMIZED MODEL (Test Set)', final_model, X_train_sc, X_test_sc, y_train, y_test)

print(f"\nFinal model parameters: {grid_search.best_params_}")
print(f"Final 5-fold CV accuracy: {grid_search.best_score_*100:.2f}%")

## Conclusion & Model Comparison

| Stage | CV Accuracy | Notes |
|---|---|---|
| Baseline (default RF) | 76.38% ± 2.14% | Untuned, default hyperparameters |
| Grid Search Tuned | **78.01% ± 2.58%** | Best params: max_depth=6, max_features=log2, min_samples_split=5, n_estimators=100 |
| Random Search Tuned | 78.18% ± 2.38% | Best params: n_estimators=200, min_samples_split=5, min_samples_leaf=5, max_features=log2, max_depth=6 |

**Improvement: +1.63 to +1.80 percentage points in CV accuracy after tuning.**

### Key Findings
1. Both Glucose and BMI had significant missing values encoded as 0 — replacing with median-imputed values was essential
2. Grid Search and Random Search converged on similar optimal regions (max_depth≈6, max_features=log2), validating the tuning process
3. The bias-variance curve shows clear underfitting below depth 3 and overfitting beyond depth 10 — depth 6 balances both
4. K=5 gave the best tradeoff between bias and variance in the CV estimate itself; K=10 had higher variance due to smaller validation folds
5. ROC-AUC improved only marginally, suggesting the dataset's predictive ceiling is close to ~80% with these features

### Recommendations
- Engineer additional features (e.g. Glucose-to-BMI ratio, Age groups) to push past the current performance ceiling
- Try Gradient Boosting or XGBoost with similar tuning for comparison
- Consider SMOTE or class weighting to address the moderate class imbalance (65% / 35%)